# HW3
## Описание датасета (Iris Species)

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cosine

df = pd.read_csv("Iris.csv")
df.head()


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [14]:
df.shape

(150, 6)

Размерность датасета - (150, 6)

In [15]:
df.dtypes

Id                 int64
SepalLengthCm    float64
SepalWidthCm     float64
PetalLengthCm    float64
PetalWidthCm     float64
Species              str
dtype: object

Все признаки, за исключением Species, числовые.

In [16]:
df.isnull().sum()

Id               0
SepalLengthCm    0
SepalWidthCm     0
PetalLengthCm    0
PetalWidthCm     0
Species          0
dtype: int64

Пропуски отсутствуют

In [17]:
classes_counts = df['Species'].value_counts()
print(classes_counts['Iris-setosa'])
print(classes_counts['Iris-versicolor'])
print(classes_counts['Iris-virginica'])

50
50
50


Распределение объектов по классам равномерное

In [18]:
df.duplicated()

0      False
1      False
2      False
3      False
4      False
       ...  
145    False
146    False
147    False
148    False
149    False
Length: 150, dtype: bool

Дубликатов нет

## Подготовка данных

Кодирование категориальных признаков

In [19]:
df_encoded = pd.get_dummies(df, columns=['Species'])
species_cols = ['Species_Iris-setosa', 'Species_Iris-versicolor', 'Species_Iris-virginica']
df_encoded['Species'] = df_encoded[species_cols].values.argmax(axis=1)
df_encoded = df_encoded.drop(columns=species_cols)
df_encoded.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,0
1,2,4.9,3.0,1.4,0.2,0
2,3,4.7,3.2,1.3,0.2,0
3,4,4.6,3.1,1.5,0.2,0
4,5,5.0,3.6,1.4,0.2,0


Выделение данных из датафрейма

In [20]:
X = df[['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']].values
y = df_encoded['Species'].values

Разбиение выборки на тестовую и тренировочную

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Масштабирование признаков

In [22]:
#scaler = StandardScaler()
#X_train = scaler.fit_transform(X_train)
#X_test = scaler.fit_transform(X_test)

Масштабирование важно для KNN, чтобы избежать искажения расстояний за счет признаков, которые численно больше, нежели другие.

Подбор гиперпараметров на тестовой выборке не следует совершать, так как тестовая выборка - это то, как поведет себя модель в незнакомых условиях

## Обучение KNN

In [23]:
class KNN():
    def __init__(self, n_neighbors=3, metric='euclidean', weights='uniform', p=2):
        self.k = n_neighbors
        self.metric_name = metric
        self.weights = weights
        self.p = p

        self.distance = {
            'euclidean' : lambda x, y: np.sqrt(np.sum((x - y)**2)),
            'manhattan' : lambda x, y: np.sum(np.abs(x - y)),
            'minkowski' : lambda x, y: np.power(np.sum(np.power(np.abs(x - y), self.p)), 1 / self.p),
            'cosine': lambda x, y: cosine(x, y)
        }[metric]
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        self.classes = np.unique(y)
        return self
    
    def predict(self, X):
        predictions = []
        for x in X:
            distances = np.array([self.distance(x, x_train) for x_train in self.X_train])
            nearest_indices = distances.argsort()[:self.k]
            nearest_labels = self.y_train[nearest_indices]

            if self.weights == 'uniform':
                predictions.append(np.argmax(np.bincount(nearest_labels)))
            elif self.weights == 'distance':
                nearest_distances = distances[nearest_labels]
                weights = np.ones_like(nearest_distances)
                non_zero_distances = nearest_distances != 0
                weights[non_zero_distances] = 1.0 / nearest_distances[non_zero_distances]
                weighted_votes = np.zeros(len(self.classes))
                for i, label in enumerate(nearest_labels):
                    weighted_votes[label] += weights[i]

                predictions.append(np.argmax(weighted_votes))
        return np.array(predictions)
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)

## Подбор гиперпараметров

In [24]:
k_values = [1, 3, 5, 7]
metrics = ['euclidean', 'manhattan', 'minkowski']
weights_options = ['uniform', 'distance']
results = []

for k in k_values:
    for metric in metrics:
        for weights in weights_options:
            knn = KNN(k, metric, weights, p=3 if metric == 'minkowski' else 2)
            knn.fit(X_train, y_train)
            accuracy = knn.score(X_test, y_test)
            results.append({
                'k' : k,
                'metric': metric,
                'weights' : weights,
                'accuracy' : accuracy
            })
            
df_results = pd.DataFrame(results)
print(df_results)

    k     metric   weights  accuracy
0   1  euclidean   uniform  1.000000
1   1  euclidean  distance  1.000000
2   1  manhattan   uniform  1.000000
3   1  manhattan  distance  1.000000
4   1  minkowski   uniform  1.000000
5   1  minkowski  distance  1.000000
6   3  euclidean   uniform  1.000000
7   3  euclidean  distance  0.966667
8   3  manhattan   uniform  1.000000
9   3  manhattan  distance  0.966667
10  3  minkowski   uniform  1.000000
11  3  minkowski  distance  0.966667
12  5  euclidean   uniform  1.000000
13  5  euclidean  distance  0.966667
14  5  manhattan   uniform  1.000000
15  5  manhattan  distance  0.933333
16  5  minkowski   uniform  0.966667
17  5  minkowski  distance  0.900000
18  7  euclidean   uniform  0.966667
19  7  euclidean  distance  0.933333
20  7  manhattan   uniform  1.000000
21  7  manhattan  distance  1.000000
22  7  minkowski   uniform  1.000000
23  7  minkowski  distance  0.900000


В моем случае, если не масштабировать признаки, показатель качества часто лучше, возможно, потому что важен именно физический размер объектов, так как это цветки ириса

## Выводы

- метод KNN хорошо срабатывает на данном датасете (точность в пределах 97%)
- параметр k = 3-5 оптимален
- подходит любая метрика расстояния, но лучше всего себя показывает манхэттенская
- масштабировать признаки нет необходимости